[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/tabular-ml-practice/04_dnn_keras/04_dnn_keras_solutions.ipynb)

# 04. 정형 데이터 신경망 — 연습 문제 해설

[04_dnn_keras.ipynb](04_dnn_keras.ipynb) 끝의 연습 문제 6개에 대한 정답 코드와 해설입니다.
**먼저 직접 시도해본 뒤** 참고하세요.

본문과 같은 순서입니다. **문제 1~4는 1부(택시·회귀), 문제 5~6은 2부(타이타닉·분류)** 범위입니다.

> 신경망은 실행할 때마다 결과가 조금씩 달라집니다. 아래 숫자는 대략적인 기준이며,
> 여러분의 실행 결과와 소수점 이하가 다른 것은 정상입니다.
> **몇몇 문제는 "예상과 반대되는" 결과가 나오는데, 그것 자체가 이 해설의 요점입니다.**

> **읽는 법** — 셀은 위에서부터 순서대로 실행해야 합니다(`Shift + Enter`). 실행 결과는 저장되어
> 있지 않으니 직접 실행해야 표와 그래프가 나타납니다. 맨 위의 **준비 셀들을 먼저 실행한 뒤**
> 원하는 문제로 건너뛰면 됩니다. 해설에 적힌 숫자는 실행하면 나오는 값입니다.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pandas seaborn matplotlib scikit-learn koreanize-matplotlib

### 준비 셀

아래 셀들은 본문과 같은 준비 코드입니다. **내용을 이해할 필요 없이 그대로 실행**하면 됩니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# 그래프에 한글이 깨지지 않도록 폰트를 설정합니다.
# koreanize-matplotlib이 있으면 그걸 쓰고, 없으면 OS에 설치된 한글 폰트를 찾습니다.
try:
    import koreanize_matplotlib  # noqa: F401
except ImportError:
    import matplotlib.font_manager as fm

    for _name in ["Malgun Gothic", "AppleGothic", "NanumGothic"]:
        if any(_name == f.name for f in fm.fontManager.ttflist):
            plt.rc("font", family=_name)
            break
plt.rcParams["axes.unicode_minus"] = False  # 한글 폰트에서 마이너스 기호가 깨지는 것 방지

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42

TensorFlow를 불러오고 시드를 고정합니다.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow:", tf.__version__)

데이터를 불러옵니다.

In [ ]:
trips = sns.load_dataset("taxis")
trips["pickup"] = pd.to_datetime(trips["pickup"])
trips["dropoff"] = pd.to_datetime(trips["dropoff"])
trips["duration"] = (trips["dropoff"] - trips["pickup"]).dt.total_seconds() / 60
trips["speed"] = trips["distance"] / (trips["duration"] / 60)
trips["weekday"] = trips["pickup"].dt.dayofweek
trips["hour"] = trips["pickup"].dt.hour

titanic = sns.load_dataset("titanic")

print("trips  :", trips.shape)
print("titanic:", titanic.shape)

02번에서 만든 전처리 함수 두 개입니다.

In [ ]:
def prepare_trips(raw):
    """[회귀] 택시 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    df = raw[(raw["duration"] > 0) & (raw["speed"] < 60)].copy()   # 이상치 제거
    df = df.drop(columns=["pickup", "dropoff",                     # 시각 자체는 weekday/hour로 대체
                          "pickup_zone", "dropoff_zone",           # 범주가 200개 이상이라 제외
                          "speed",                                 # duration으로 계산한 값 → 정답 누출
                          "total"])                                # fare+tip+tolls의 합 → 중복
    df = df.dropna()                                               # 결측치 행 제거
    df = pd.get_dummies(df, columns=["color", "payment",
                                     "pickup_borough", "dropoff_borough"],
                        drop_first=True)                           # 범주형 → 0/1
    X = df.drop(columns="duration")
    y = df["duration"]
    return X, y


def prepare_titanic(raw):
    """[분류] 타이타닉 데이터를 학습 가능한 형태로 만든다. 02_preprocessing에서 단계별로 만든 코드."""
    q1, q3 = raw["fare"].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    df = raw[(raw["fare"] >= lower) & (raw["fare"] <= upper)].copy()  # 이상치 제거
    df = df.drop(columns=["alive",                                   # survived와 같은 정보 → 정답 누출
                          "class", "embark_town",                    # pclass/embarked와 중복
                          "deck",                                    # 결측치가 77%
                          "adult_male"])                             # who와 중복
    df = df.dropna()
    df = pd.get_dummies(df, columns=["sex", "embarked", "who"], drop_first=True)
    X = df.drop(columns="survived")
    y = df["survived"]
    return X, y

분리와 스케일링까지 본문과 똑같이 준비합니다.
**문제 1~4는 회귀(`X_train_s`), 문제 5~6은 분류(`Xc_train_s`)** 를 씁니다.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

X_reg, y_reg = prepare_trips(trips)
X_reg = X_reg.drop(columns=["fare", "tip", "tolls"])
X_clf, y_clf = prepare_titanic(titanic)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)
Xc_train, Xc_valid, yc_train, yc_valid = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_STATE, stratify=y_clf
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype("float32")
X_valid_s = scaler.transform(X_valid).astype("float32")

scaler_c = StandardScaler()
Xc_train_s = scaler_c.fit_transform(Xc_train).astype("float32")
Xc_valid_s = scaler_c.transform(Xc_valid).astype("float32")

y_train_a = y_train.values.astype("float32")
y_valid_a = y_valid.values.astype("float32")
yc_train_a = yc_train.values.astype("float32")
yc_valid_a = yc_valid.values.astype("float32")

print(f"[회귀] 학습 {X_train_s.shape}  검증 {X_valid_s.shape}")
print(f"[분류] 학습 {Xc_train_s.shape}  검증 {Xc_valid_s.shape}")

---

# 1부 — 택시 (회귀)

## 문제 1. 모델 크기를 바꿔가며 비교

In [ ]:
def build_and_train(units, tag, epochs=200):
    tf.random.set_seed(42)
    model = keras.Sequential(
        [layers.Input(shape=(X_train_s.shape[1],))]
        + [layers.Dense(u, activation="relu") for u in units]
        + [layers.Dense(1)]
    )
    model.compile(optimizer="adam", loss="mse", metrics=["mae"])

    hist = model.fit(
        X_train_s, y_train_a, validation_data=(X_valid_s, y_valid_a),
        epochs=epochs, batch_size=128, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                                 restore_best_weights=True)],
    )
    pred = model.predict(X_valid_s, verbose=0).ravel()
    return {
        "구조": tag,
        "파라미터": model.count_params(),
        "학습 epoch": len(hist.history["loss"]),
        "검증 MAE": mean_absolute_error(y_valid_a, pred),
    }, hist


results = []
histories = {}
for units, tag in [([16], "Dense(16)"),
                   ([64, 32, 16], "64-32-16"),
                   ([256, 128, 64, 32], "256-128-64-32")]:
    row, hist = build_and_train(units, tag)
    results.append(row)
    histories[tag] = hist

pd.DataFrame(results).set_index("구조").round(4)

세 모델의 학습 곡선을 겹쳐 그려 **어느 쪽이 먼저 꺾이는지** 봅니다.

In [ ]:
plt.figure(figsize=(10, 5))
for tag, hist in histories.items():
    plt.plot(hist.history["val_mae"], label=tag)
plt.xlabel("Epoch")
plt.ylabel("검증 MAE")
plt.title("모델 크기별 학습 곡선")
plt.ylim(3, 8)
plt.legend()
plt.show()

**해설 — 큰 모델이 더 좋지 않습니다**

| 구조 | 파라미터 | 학습 epoch | 검증 MAE |
|---|---|---|---|
| `Dense(16)` | **241** | 200 (끝까지) | 3.573 |
| **`64-32-16`** | 3,521 | 59 | **3.517** |
| `256-128-64-32` | **46,849** | **28** | **3.668 ← 가장 나쁨** |

**파라미터가 194배 많은 모델이 가장 나쁩니다.**

- **가장 작은 모델(241개)**: 200 epoch을 끝까지 다 썼습니다. `EarlyStopping`이 걸리지 않았다는 것은
  **아직 더 배울 여지가 있었다**는 뜻입니다. 표현력이 부족해 천천히 나아지고 있었습니다 (**과소적합**)
- **중간 모델**: 59 epoch에서 멈췄습니다. 적절한 지점입니다
- **가장 큰 모델**: **28 epoch만에 멈췄습니다.** 검증 손실이 가장 빨리 나빠졌다는 뜻이고,
  곧 **가장 빨리 과적합**했다는 뜻입니다

학습 데이터가 5,068건인데 파라미터가 46,849개입니다. **데이터 한 건당 파라미터 9개**로,
외우는 것 말고 할 일이 없습니다.

> **표 데이터에서 모델을 키우는 것은 대개 손해입니다.** 이미지·언어 모델에서 "크면 좋다"는
> 경험칙이 통하는 이유는 데이터가 수백만~수십억 건이기 때문입니다. 수천 건짜리 표 데이터에서는
> **층 2~3개, 뉴런 수십 개** 규모에서 시작해 필요할 때만 늘리는 것이 맞습니다.
>
> 그리고 세 모델의 MAE 차이가 0.15분(9초)에 불과합니다. **구조를 바꿔서 얻는 이득은
> 생각보다 작습니다.** 03번 연습문제 2번에서 이야기했듯, 같은 시간을 **피처를 개선하는 데**
> 쓰는 편이 대개 더 큰 이득을 줍니다.

## 문제 2. 타깃 로그 변환

03번 연습문제 2번에서 **"긴 운행을 9.5분이나 짧게 예측한다"** 는 문제를 발견하고,
개선안으로 로그 변환을 제안했습니다. **정말 효과가 있는지 확인합니다.**

In [ ]:
def train_reg(y_tr, y_va, tag, inverse=None):
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(X_train_s.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1),
    ])
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    m.fit(X_train_s, y_tr, validation_data=(X_valid_s, y_va),
          epochs=200, batch_size=128, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                                   restore_best_weights=True)])
    pred = m.predict(X_valid_s, verbose=0).ravel()
    if inverse is not None:
        pred = inverse(pred)
    print(f"{tag} 학습 완료")
    return pred


pred_plain = train_reg(y_train_a, y_valid_a, "원본 타깃")
# log1p(x)=log(1+x), expm1은 그 역함수. 0을 포함한 값에도 안전하게 로그를 씌우고 되돌린다
pred_log = train_reg(np.log1p(y_train_a), np.log1p(y_valid_a), "로그 변환", inverse=np.expm1)

왜 로그 변환이 도움이 될 수 있는지, 분포가 어떻게 바뀌는지 먼저 눈으로 확인합니다.

In [ ]:
# 로그 변환이 분포를 어떻게 바꾸는지 먼저 확인
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y_train_a, bins=50, ax=axes[0])
axes[0].set_title("원본 duration — 오른쪽으로 치우침")
sns.histplot(np.log1p(y_train_a), bins=50, ax=axes[1])
axes[1].set_title("log1p(duration) — 좌우대칭에 가까움")
plt.tight_layout()
plt.show()

두 예측을 한 표에 모아 **절대 오차·상대 오차·잔차**를 계산합니다.

In [ ]:
comp = pd.DataFrame({"실제": y_valid_a, "원본": pred_plain, "로그": pred_log})
comp["원본_절대오차"] = (comp["실제"] - comp["원본"]).abs()
comp["로그_절대오차"] = (comp["실제"] - comp["로그"]).abs()
comp["원본_잔차"] = comp["실제"] - comp["원본"]
comp["로그_잔차"] = comp["실제"] - comp["로그"]
comp["원본_상대오차"] = comp["원본_절대오차"] / comp["실제"] * 100
comp["로그_상대오차"] = comp["로그_절대오차"] / comp["실제"] * 100

print("=== 전체 ===")
print(f"  MAE  (절대 오차, 분)  원본 {comp['원본_절대오차'].mean():.3f}   로그 {comp['로그_절대오차'].mean():.3f}")
print(f"  MAPE (상대 오차, %)   원본 {comp['원본_상대오차'].mean():.2f}   로그 {comp['로그_상대오차'].mean():.2f}")

전체 평균만 보면 결론이 하나로 나오지 않습니다. **구간별로 나눠야** 어디서 좋아지고 어디서
나빠졌는지 드러납니다.

In [ ]:
comp["구간"] = pd.cut(comp["실제"], [0, 10, 20, 40, 120],
                      labels=["~10분", "10-20분", "20-40분", "40분+"])

comp.groupby("구간", observed=True).agg(
    건수=("실제", "size"),
    원본_MAE=("원본_절대오차", "mean"),
    로그_MAE=("로그_절대오차", "mean"),
    원본_상대오차=("원본_상대오차", "mean"),
    로그_상대오차=("로그_상대오차", "mean"),
    원본_잔차=("원본_잔차", "mean"),
    로그_잔차=("로그_잔차", "mean"),
).round(2)

표를 그래프로 확인합니다. 왼쪽은 구간별 잔차(0에 가까울수록 편향 없음), 오른쪽은 상대 오차입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

melted = comp.melt(id_vars="구간", value_vars=["원본_잔차", "로그_잔차"],
                   var_name="모델", value_name="잔차")
sns.boxplot(data=melted, x="구간", y="잔차", hue="모델", ax=axes[0])
axes[0].axhline(0, color="red", linestyle="--", linewidth=1)
axes[0].set_title("구간별 잔차 (0에 가까울수록 편향 없음)")

melted2 = comp.melt(id_vars="구간", value_vars=["원본_상대오차", "로그_상대오차"],
                    var_name="모델", value_name="상대오차(%)")
sns.barplot(data=melted2, x="구간", y="상대오차(%)", hue="모델", errorbar=None, ax=axes[1])
axes[1].set_title("구간별 상대 오차")

plt.tight_layout()
plt.show()

**해설 — 제안했던 개선안이 절반만 통했습니다**

### 전체 결과

| | 원본 타깃 | 로그 변환 |
|---|---|---|
| **MAE** (절대 오차) | **3.476분** | 3.660분 ← **나빠짐** |
| **MAPE** (상대 오차) | 32.22% | **28.91%** ← **좋아짐** |

**절대 오차는 나빠지고 상대 오차는 좋아졌습니다.** 예상과 다릅니다.

### 구간별로 보면 이유가 분명합니다

| 구간 | 건수 | 원본 MAE | 로그 MAE | 원본 잔차 | 로그 잔차 |
|---|---|---|---|---|---|
| ~10분 | 601 | 2.07 | **1.85** | -1.25 | **-0.70** |
| 10-20분 | 399 | **3.42** | 3.49 | -0.35 | 0.16 |
| 20-40분 | 217 | **5.74** | 6.69 | 0.76 | 0.69 |
| **40분+** | 51 | **10.79** | 13.39 | **+8.48** | **+4.73** |

**① 의도한 효과는 보였습니다 — 이 실행에서는 편향이 절반으로 줄었습니다**

40분 이상 구간의 잔차 평균이 **+8.48 → +4.73**. "긴 운행을 체계적으로 짧게 예측한다"는
문제가 완화되었습니다. 로그 스케일에서는 40분을 30분으로 예측하는 것이
5분을 3.75분으로 예측하는 것과 같은 크기의 잘못이 되므로, 모델이 긴 운행을 더 신경 씁니다.

> **다만 이 개선은 실행마다 재현되지 않습니다.** 시드를 바꿔 여러 번 돌려보면 40분+ 구간의 잔차가
> 줄어드는 경우도, 오히려 늘어나는 경우도 나옵니다. 해당 구간이 51건뿐이라 몇 건의 예측이
> 평균을 좌우하기 때문입니다. **안정적으로 재현되는 것은 아래 ②·③ — 전체 MAE는 나빠지고
> MAPE와 짧은 운행 구간은 좋아진다는 쪽**입니다. 편향 개선 여부를 결론으로 삼으려면
> 시드를 바꿔 여러 번 돌린 평균을 봐야 합니다.

**② 그런데 그 구간의 절대 오차는 오히려 커졌습니다 (10.79 → 13.39)**

편향(평균적으로 치우침)은 줄었지만 **분산(들쭉날쭉함)이 커졌습니다.** 예측을 위로 끌어올리면서
어떤 건은 과하게 올라가고 어떤 건은 여전히 모자란 것입니다.

이것이 통계에서 말하는 **편향-분산 트레이드오프**의 한 사례입니다.

**③ 대신 짧은 운행이 좋아졌습니다**

~10분 구간에서 MAE 2.07 → 1.85, 상대 오차 42.2% → 33.5%. 601건으로 **가장 건수가 많은
구간**이라 전체 MAPE 개선에 크게 기여했습니다.

### 그래서 어느 쪽이 맞나

**목적에 따라 다릅니다.**

- **"평균 몇 분 틀리는가"가 기준이라면** → 원본 타깃 (MAE 3.48)
- **"몇 % 틀리는가"가 기준이라면** → 로그 변환 (MAPE 28.9%)
- **"공항 가는 손님에게 정확한 시간을 주는 것"이 목표라면** → 로그 변환이 편향을 줄이므로 유리

도착 시간 안내 서비스라면 **상대 오차가 더 자연스러운 기준**일 수 있습니다.
5분 걸릴 길을 7분이라 하는 것(40% 오차)과 50분 걸릴 길을 52분이라 하는 것(4% 오차)은
체감이 완전히 다릅니다.

### 이 문제의 진짜 교훈

**제안한 개선안은 반드시 측정해봐야 합니다.** 03번에서 "로그 변환하면 좋아질 것"이라고
썼지만, 실제로는 지표에 따라 좋아지기도 나빠지기도 했습니다.

그리고 **여전히 40분+ 구간의 잔차는 +4.73분**입니다. 근본 원인이 남아 있다는 뜻입니다 —
**모델에 "이 운행이 공항행인지" 알려주는 정보가 아예 없습니다.**
타깃 변환은 증상 완화이고, 근본 해결은 **피처 추가**입니다
(02번 연습문제 3번의 zone 활용 방법을 참고하세요).

## 문제 3. 학습이 진행되지 않는 코드

**문제로 주어진 코드**

```python
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),      # ← 문제 1
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])   # ← 문제 2
model.fit(X_train_s, y_train_a, epochs=30, batch_size=128, verbose=0)
```

In [ ]:
# 실제로 어떻게 되는지 확인해봅시다
tf.random.set_seed(42)
broken = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
broken.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
h_broken = broken.fit(X_train_s, y_train_a, epochs=30, batch_size=128, verbose=0)

pred_broken = broken.predict(X_valid_s, verbose=0).ravel()

print(f"첫 epoch 손실   : {h_broken.history['loss'][0]:.4f}")
print(f"마지막 손실     : {h_broken.history['loss'][-1]:.4f}")
print()
print(f"예측값의 범위   : {pred_broken.min():.4f} ~ {pred_broken.max():.4f}")
print(f"실제 정답의 범위: {y_valid_a.min():.2f} ~ {y_valid_a.max():.2f}")
print()
print(f"MAE: {mean_absolute_error(y_valid_a, pred_broken):.4f} 분")

무엇이 문제인지 확인했으니, **회귀용 설정**으로 바꿔 다시 학습해봅니다.

In [ ]:
# 고친 버전
tf.random.set_seed(42)
fixed = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),                                    # 활성화 함수 없음
])
fixed.compile(optimizer="adam", loss="mse", metrics=["mae"])   # 회귀용 손실 함수
h_fixed = fixed.fit(X_train_s, y_train_a, validation_data=(X_valid_s, y_valid_a),
                    epochs=30, batch_size=128, verbose=0)

pred_fixed = fixed.predict(X_valid_s, verbose=0).ravel()

print(f"예측값의 범위: {pred_fixed.min():.2f} ~ {pred_fixed.max():.2f}")
print(f"MAE: {mean_absolute_error(y_valid_a, pred_fixed):.4f} 분")

**해설 — 회귀 문제에 분류용 설정을 썼습니다**

`y_train_a`는 **이동 시간(분)** 입니다. 0.1분부터 107분까지의 연속값이죠. 그런데 코드는
**이진 분류용 설정**을 쓰고 있습니다.

### 문제 ①: 출력층의 `activation="sigmoid"`

`sigmoid`는 어떤 입력이 들어와도 **출력을 0과 1 사이로 눌러버립니다.**

```
sigmoid(-100) ≈ 0.000
sigmoid(0)    = 0.500
sigmoid(100)  ≈ 1.000
```

위 실행 결과에서 예측값의 범위가 **0~1 안에 갇혀 있는 것**을 확인할 수 있습니다.
정답은 0.1~107분인데 예측은 최대 1입니다. **구조적으로 맞힐 수가 없습니다.**

### 문제 ②: `loss="binary_crossentropy"`

BCE는 **정답이 0 또는 1일 때** 쓰도록 만들어진 손실 함수입니다.

```
BCE = -[y·log(p) + (1-y)·log(1-p)]
```

`y = 14.3`(분)을 넣으면 수식이 의미를 잃습니다. 계산은 되지만(그래서 **에러가 나지 않습니다**)
"이동 시간이 정답에 가까워지는 방향"과는 아무 상관 없는 값이 나옵니다.
모델은 **엉뚱한 목표를 향해 성실히 학습**합니다.

위 실행 결과에서 **손실이 -27에서 -779,620으로 발산**한 것을 보세요. BCE는 정의상
**항상 0 이상**이어야 하는 값입니다. `y`가 1보다 크면 `-y·log(p)` 항이 음수로 커질 수 있어
이런 일이 벌어집니다. **손실이 음수이거나 무한정 커진다면 손실 함수를 잘못 골랐다는
거의 확실한 신호**입니다.

### 문제 ③(부수적): `metrics=["accuracy"]`

연속값에 정확도는 의미가 없습니다. "예측이 정답과 정확히 일치한 비율"이 되는데,
14.30분과 14.31분은 불일치로 셉니다. **거의 항상 0**이 나옵니다.

### 고친 코드

```python
model = keras.Sequential([
    layers.Input(shape=(X_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(32, activation="relu"),
    layers.Dense(1),                      # ✅ 활성화 함수 없음 (linear)
])
model.compile(optimizer="adam",
              loss="mse",                 # ✅ 회귀용 손실
              metrics=["mae"])            # ✅ 회귀용 지표
model.fit(X_train_s, y_train_a,
          validation_data=(X_valid_s, y_valid_a),   # ✅ 검증 데이터도 넣기
          epochs=30, batch_size=128, verbose=0)
```

**이 실수가 무서운 이유는 에러가 나지 않는다는 것입니다.** 손실 값이 출력되고 학습도
진행되는 것처럼 보입니다. **`validation_data`를 넣지 않았다면** 검증 손실조차 볼 수 없어
이상한 점을 눈치채기 더 어려웠을 것입니다.

**진단 습관**: 학습이 이상하면 **예측값을 직접 출력해서 정답과 범위를 비교**해보세요.
`pred.min(), pred.max()` 두 줄이면 이 문제는 즉시 드러납니다.

> 본문 4절의 표를 다시 확인하세요. **출력층 활성화 + 손실 함수는 문제 유형에 따라
> 한 세트로 정해집니다.** 이 조합을 틀리는 것이 Keras에서 가장 흔한 실수입니다.

## 문제 4. `BatchNormalization`의 효과

In [ ]:
def build_with_bn(use_bn, tag):
    tf.random.set_seed(42)
    layer_list = [layers.Input(shape=(X_train_s.shape[1],))]
    for u in [64, 32]:
        layer_list.append(layers.Dense(u))
        if use_bn:
            # BatchNormalization: 층의 출력을 배치 단위로 평균 0·분산 1로 정규화한다
            layer_list.append(layers.BatchNormalization())
        layer_list.append(layers.Activation("relu"))
    layer_list.append(layers.Dense(1))

    m = keras.Sequential(layer_list)
    m.compile(optimizer="adam", loss="mse", metrics=["mae"])
    h = m.fit(X_train_s, y_train_a, validation_data=(X_valid_s, y_valid_a),
              epochs=60, batch_size=128, verbose=0)
    return m, h


model_plain, hist_plain = build_with_bn(False, "없음")
model_bn, hist_bn = build_with_bn(True, "BatchNorm")

summary = []
for tag, m, h in [("BatchNorm 없음", model_plain, hist_plain),
                  ("BatchNorm 있음", model_bn, hist_bn)]:
    v = np.array(h.history["val_mae"])
    reached = next((i + 1 for i, x in enumerate(v) if x < 4.0), None)
    summary.append({
        "모델": tag,
        "파라미터": m.count_params(),
        "val_mae<4.0 도달": reached,
        "최종 val_mae": v[-1],
        "최소 val_mae": v.min(),
        "후반 변동성(std)": v[-20:].std(),
    })

pd.DataFrame(summary).set_index("모델").round(4)

두 곡선을 겹쳐 그리고, 오른쪽에서 후반부를 확대해 **흔들림의 크기**를 비교합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_plain.history["val_mae"], label="BatchNorm 없음")
axes[0].plot(hist_bn.history["val_mae"], label="BatchNorm 있음")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("검증 MAE")
axes[0].set_title("전체 학습 곡선")
axes[0].legend()

axes[1].plot(hist_plain.history["val_mae"], label="BatchNorm 없음")
axes[1].plot(hist_bn.history["val_mae"], label="BatchNorm 있음")
axes[1].set_xlim(10, 60)
axes[1].set_ylim(3.3, 4.2)
axes[1].set_xlabel("Epoch")
axes[1].set_title("후반부 확대")
axes[1].legend()

plt.tight_layout()
plt.show()

**해설 — 이 상황에서는 도움이 되지 않았습니다**

| | 없음 | 있음 |
|---|---|---|
| 파라미터 | 3,009 | 3,393 |
| `val_mae < 4.0` 도달 | 14 epoch | 14 epoch |
| 최종 `val_mae` | **3.491** | 3.543 |
| 후반 변동성(표준편차) | **0.004** | **0.028** |

**수렴 속도는 비슷하고, 최종 성능은 조금 나쁘고, 곡선은 오히려 더 출렁입니다.**
(변동성 차이의 배수는 실행마다 크게 달라집니다. 몇 배가 나오든 **BatchNorm을 넣어서
좋아지지 않았다**는 방향만 확인하면 됩니다.)

### `BatchNormalization`이 하는 일

각 층의 출력을 **배치 단위로 평균 0, 분산 1로 정규화**한 뒤, 학습 가능한 파라미터
(`gamma`, `beta`)로 다시 스케일·이동시킵니다. 층마다 스케일링을 한 번 더 하는 셈입니다.

원래 목적은 **내부 공변량 변화(internal covariate shift)** 완화입니다. 깊은 신경망에서는
앞쪽 층의 가중치가 조금만 바뀌어도 뒤쪽 층이 받는 입력 분포가 크게 흔들리는데,
매 층에서 정규화하면 이 흔들림이 줄어 학습이 안정됩니다.

### 왜 여기서는 효과가 없었나

1. **층이 2개뿐입니다.** 정규화로 얻을 안정화 효과가 애초에 크지 않습니다.
   `BatchNormalization`은 **층이 10개, 50개씩 되는 깊은 신경망**에서 진가를 발휘합니다
2. **입력이 이미 `StandardScaler`로 정규화되어 있습니다.** 첫 층이 받는 분포가 이미 좋습니다
3. **배치 단위 통계를 쓰기 때문에 노이즈가 추가됩니다.** 배치 128개의 평균·분산은 매번
   조금씩 다르고, 이것이 후반 변동성이 커진 원인입니다
4. **학습과 예측의 동작이 다릅니다.** 학습 때는 현재 배치의 통계를, 예측 때는 학습 내내
   누적한 이동 평균을 씁니다. 이 차이가 작은 모델에서는 손해로 나타날 수 있습니다

### 언제 쓰면 좋은가

| 상황 | `BatchNormalization` |
|---|---|
| 층 2~3개의 얕은 표 데이터 모델 | **불필요** |
| 층이 깊은 신경망 (10층 이상) | **유용** |
| CNN | **거의 표준** |
| 배치 크기가 작을 때 (< 16) | 통계가 불안정해 **오히려 해로움** (`LayerNormalization` 권장) |

> 문제 1의 결론과 이어집니다. **표 데이터에서는 "딥러닝의 표준 기법"이 그대로 통하지
> 않는 경우가 많습니다.** 모델을 키우는 것도, `BatchNormalization`을 넣는 것도
> 여기서는 도움이 되지 않았습니다. 기법을 관성적으로 넣지 말고 **넣기 전과 후를
> 측정해서 판단하는 습관**이 중요합니다.

---

# 2부 — 타이타닉 (분류)

## 문제 5. `sigmoid` 1개 vs `softmax` 2개

In [ ]:
from tensorflow.keras.utils import to_categorical   # to_categorical: 정수 라벨(0/1)을 원-핫([1,0]/[0,1])으로 바꾼다

# ① sigmoid + binary_crossentropy (본문 방식)
tf.random.set_seed(42)
model_sig = keras.Sequential([
    layers.Input(shape=(Xc_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model_sig.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model_sig.fit(Xc_train_s, yc_train_a, validation_data=(Xc_valid_s, yc_valid_a),
              epochs=300, batch_size=32, verbose=0,
              callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                                       restore_best_weights=True)])

pred_sig = (model_sig.predict(Xc_valid_s, verbose=0).ravel() >= 0.5).astype(int)
print(f"sigmoid(1개): 파라미터 {model_sig.count_params()}, "
      f"정확도 {accuracy_score(yc_valid_a, pred_sig):.4f}")

같은 문제를 `softmax` 2개로 풀려면 **라벨을 원-핫으로** 바꿔야 합니다.

In [ ]:
# ② softmax + categorical_crossentropy — 라벨을 원-핫으로 변환해야 합니다
Y_train_oh = to_categorical(yc_train_a.astype(int), num_classes=2)
Y_valid_oh = to_categorical(yc_valid_a.astype(int), num_classes=2)

print("원본 라벨 :", yc_train_a[:5])
print("원-핫 라벨:")
print(Y_train_oh[:5])

출력층만 `Dense(2, activation="softmax")`로 바꾼 모델입니다. 은닉층은 그대로입니다.

In [ ]:
tf.random.set_seed(42)
model_soft = keras.Sequential([
    layers.Input(shape=(Xc_train_s.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),
    layers.Dense(2, activation="softmax"),        # 뉴런 2개
])
model_soft.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model_soft.fit(Xc_train_s, Y_train_oh, validation_data=(Xc_valid_s, Y_valid_oh),
               epochs=300, batch_size=32, verbose=0,
               callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                                        restore_best_weights=True)])

pred_soft = model_soft.predict(Xc_valid_s, verbose=0).argmax(axis=1)   # 확률이 가장 큰 클래스
print(f"softmax(2개): 파라미터 {model_soft.count_params()}, "
      f"정확도 {accuracy_score(yc_valid_a, pred_soft):.4f}")

두 방식의 출력이 실제로 어떻게 생겼는지 비교해봅시다.

In [ ]:
# softmax 출력은 두 클래스의 확률이고, 합이 항상 1입니다
probs = model_soft.predict(Xc_valid_s, verbose=0)

print("softmax 출력 예시 (사망 확률, 생존 확률):")
print(probs[:5].round(4))
print("\n각 행의 합:", probs[:5].sum(axis=1).round(6))
print()
print("sigmoid 출력 예시 (생존 확률 하나만):")
print(model_sig.predict(Xc_valid_s, verbose=0).ravel()[:5].round(4))

**해설 — 수학적으로 같은 모델입니다**

| 방식 | 출력 뉴런 | 파라미터 | 라벨 형태 | 예측 방법 |
|---|---|---|---|---|
| `sigmoid` | 1 | 2,881 | `[0, 1, 1, ...]` | `>= 0.5` |
| `softmax` | 2 | 2,914 | `[[1,0], [0,1], ...]` | `argmax` |

**두 방식은 이진 분류에서 수학적으로 동등합니다.**

`softmax`를 클래스가 2개일 때 전개하면

```
softmax(z)₁ = e^z₁ / (e^z₀ + e^z₁)
            = 1 / (1 + e^(z₀ - z₁))
            = sigmoid(z₁ - z₀)
```

즉 **두 출력의 차이에 `sigmoid`를 적용한 것과 같습니다.** `softmax` 쪽은 출력이 2개라
파라미터가 33개(= 32×1 + 1) 더 많지만, 하나가 다른 하나로 완전히 결정되는 잉여 파라미터입니다.
02번의 **더미 변수 함정**과 정확히 같은 구조입니다.

**성능 차이는 실행 오차입니다.** 위 결과에서 정확도가 다르게 나왔다면, 그것은 방식의 차이가
아니라 **가중치 초기화와 Dropout의 무작위성** 때문입니다. 검증 데이터가 184건이라
2~6명 차이면 3%p가 움직입니다. 여러 번 실행하면 순서가 뒤바뀝니다.

**그러면 어느 쪽을 쓸까**

- **이진 분류라면 `sigmoid` 1개**를 권합니다. 파라미터가 적고, 출력이 곧 확률이라
  임계값 조정(03번 연습문제 5번)이 자연스럽습니다
- **클래스가 3개 이상이면 `softmax`** 외에 선택지가 없습니다
- 다중 분류에서 `to_categorical` 변환이 번거롭다면 **`sparse_categorical_crossentropy`** 를
  쓰면 정수 라벨을 그대로 넣을 수 있습니다

```python
model.compile(loss="sparse_categorical_crossentropy", ...)
model.fit(X, y_int, ...)      # to_categorical 불필요
```

## 문제 6. `restore_best_weights`의 효과

In [ ]:
rows = []
for rbw in [True, False]:
    tf.random.set_seed(42)
    m = keras.Sequential([
        layers.Input(shape=(Xc_train_s.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=20,
                                       restore_best_weights=rbw)
    h = m.fit(Xc_train_s, yc_train_a, validation_data=(Xc_valid_s, yc_valid_a),
              epochs=300, batch_size=32, callbacks=[es], verbose=0)

    pred = (m.predict(Xc_valid_s, verbose=0).ravel() >= 0.5).astype(int)
    rows.append({
        "restore_best_weights": rbw,
        "총 epoch": len(h.history["loss"]),
        "최저점 epoch": int(np.argmin(h.history["val_loss"])) + 1,
        "최저 val_loss": min(h.history["val_loss"]),
        "마지막 val_loss": h.history["val_loss"][-1],
        "검증 정확도": accuracy_score(yc_valid_a, pred),
    })

pd.DataFrame(rows).set_index("restore_best_weights").round(4)

**해설**

| | `True` | `False` |
|---|---|---|
| 총 epoch | 26 | 26 |
| 최저점 epoch | **6** | **6** |
| 최저 `val_loss` | 0.4600 | 0.4595 |
| 마지막 `val_loss` | 0.5044 | 0.5137 |
| **검증 정확도** | **0.8152** | 0.7989 |

**학습 과정은 사실상 같습니다.** 둘 다 6 epoch 근처에서 최저점을 찍고 20여 epoch 뒤에 멈췄습니다.

> **위 표의 정확도 격차는 실행 환경에 따라 달라집니다.** 두 값이 똑같이 나오거나 순서가 뒤집히는
> 경우도 있습니다. 검증 데이터가 184건이라 3명만 달라져도 1.6%p가 움직이기 때문입니다.
> **격차의 크기가 아니라 "멈춘 시점의 가중치는 최저점의 가중치보다 나쁘다"는 방향**이 요점입니다.

`restore_best_weights`는 **학습이 끝난 뒤 가중치를 어느 시점 것으로 남길지**만 결정합니다.

```
epoch  1 ─── 6 ─────────────── 26
            ▲                  ▲
        최저점             멈춘 지점
        (val_loss 0.46)   (val_loss 0.50)

  restore=True  → 6 epoch의 가중치를 남김   ✅
  restore=False → 26 epoch의 가중치를 남김  ❌
```

**`patience`가 이 격차를 만듭니다.** `patience=20`이므로 최저점 이후 **20 epoch을 더 지켜본
뒤에야** 멈춥니다. 그 20 epoch 동안 모델은 계속 과적합됩니다.
`restore_best_weights=False`면 **일부러 더 나빠지게 만든 상태의 모델**이 남습니다.

**`patience`가 클수록 손해가 커집니다.**

| `patience` | 낭비되는 epoch | 위험 |
|---|---|---|
| 5 | 5 | 일시적 출렁임에 속아 일찍 멈출 수 있음 |
| 20 | 20 | 안정적이지만 `restore` 없으면 손해가 큼 |
| 50 | 50 | `restore=False`라면 사실상 EarlyStopping을 안 쓴 것과 비슷 |

**결론: `EarlyStopping`을 쓴다면 `restore_best_weights=True`를 거의 항상 함께 씁니다.**
`patience`를 크게 잡아 안정적으로 관찰하면서도 최적 시점을 놓치지 않는 조합입니다.

> `ModelCheckpoint(save_best_only=True)`로 파일에 저장해두고 나중에 불러오는 방법도 있습니다.
> 학습이 오래 걸려 중간에 끊길 위험이 있을 때 유용합니다.

---

이것으로 `tabular-ml-practice` 시리즈가 끝납니다.
전체 흐름을 다시 보려면 [시리즈 README](https://github.com/karzit/temp/blob/master/notebooks/tabular-ml-practice/README.md)를 참고하세요.